# In Orbit Simulation of the AtmoLITE Limb Sounder for SHIPAS 

In [ ]:
from skyfield.api import load, wgs84

satellites_url = "https://celestrak.org/NORAD/elements/gp.php?GROUP=active&FORMAT=tle"
# Get all active satellites
satellites = load.tle_file(satellites_url, reload=True)
print("Loaded", len(satellites), "satellites")

## Read the Orbital Elements of a given active Satellite

In [3]:
satellite_name ="SENTINEL-5P"
by_name = {sat.name: sat for sat in satellites}
# Select a satellite by name
satellite = by_name[satellite_name]
FOV = 1.4

In [ ]:
from datetime import datetime, timezone

# Get the epoch of the satellite data, update time stamp of the last orbit measurements
print("Last upate of orbital elements:", satellite.epoch.utc_iso())
print(
    "Time different since last update:",
    (datetime.now(timezone.utc)) - satellite.epoch.utc_datetime(),
)

## Analyse Orbital Elements

In [ ]:
# https://rhodesmill.org/skyfield/api-satellites.html#api-reference-earth-satellites
from pprint import pprint
from sgp4 import exporter


exporter.export_tle(satellite.model)
fields = exporter.export_omm(satellite.model, satellite)
pprint(fields)

In [ ]:
print(
    "The satellite rotates around the earth " + str(fields["MEAN_MOTION"]) + " per Day"
)
print("One orbital period in minutes:", 24 * 60 / fields["MEAN_MOTION"])
print("The Longitude of the Ascending Node in degree:", fields["RA_OF_ASC_NODE"])
print("The Inclination is in degree:", fields["INCLINATION"])
print("Orbit Eccentricity:", fields["ECCENTRICITY"])

![](./img/Orbit1.png)

## Load JPL Planetary and Lunar Ephemerides

In astronomy and celestial navigation, an ephemeris is a book with tables that gives the trajectory of naturally occurring astronomical objects as well as artificial satellites in the sky, i.e., the position (and possibly velocity) over time.

[Ref NASA](https://ssd.jpl.nasa.gov/planets/eph_export.html)
[Ref Wikipedia](https://en.wikipedia.org/wiki/Ephemeris)

In [7]:
from skyfield import api
from skyfield.magnitudelib import planetary_magnitude

# docs: https://rhodesmill.org/skyfield/searches.html#finding-extrema

ts = api.load.timescale()
t0 = ts.now()

eph = load("de421.bsp")
earth, venus, moon, sun = eph["earth"], eph["venus"], eph["moon"], eph["sun"]

## Create Vectors from Barycenter to Earth to Satellite
[<img src="img/bary.png" width="400"/>](img/bary.png)

In [ ]:
print("Moon Vector: ")
print(moon)
print("Satellite Vector: ")
satellite_vector = earth + satellite
print(satellite_vector)
print("Satellite State: ")
satellite_state = satellite_vector.at(t0)
print(satellite_state)
print(satellite_state.velocity)

[<img src="img/Orbital_mechanics.png"/>](img/Orbital_mechanics.png)

## Create Observation Vector for Moon and the Sun

In [ ]:
moon_observation = satellite_state.observe(moon)
sun_observation = satellite_state.observe(sun)
moon_apparent = moon_observation.apparent()

print("Moon apparent: ")
print(moon_apparent)

view = moon_apparent.is_behind_earth()
print("Is the moon behind the earth?", view)
sunlit = satellite.at(t0).is_sunlit(eph)
print("Is the satellite illuminated by the sun?", sunlit)
print(
    "Moon relative velocity to the observer:", moon_apparent.velocity.km_per_s, "km/s"
)

In [ ]:
import numpy as np
from skyfield.api import Angle, load

# Source: https://ssd.jpl.nasa.gov/horizons/app.html#/
# Docs: https://github.com/skyfielders/python-skyfield/issues/517
# Docs: https://rhodesmill.org/skyfield/examples.html#what-is-the-angular-diameter-of-a-planet-given-its-radius
radius_moon_km = 1737.53
ra, dec, distance = moon_apparent.radec()
apparent_diameter_moon = Angle(radians=np.arcsin(radius_moon_km / distance.km) * 2.0)
print(
    "The Angular diameter of moon from the observer postion: {:.6f} degrees".format(
        apparent_diameter_moon.degrees
    )
)

In [ ]:
# What Angle in the sky is the crescent moon?
from skyfield.api import N, W, load, wgs84
from skyfield.trigonometry import position_angle_of


# docs: https://rhodesmill.org/skyfield/examples.html#at-what-angle-in-the-sky-is-the-crescent-moon
# https://en.wikipedia.org/wiki/Lunar_phase#Orientation_by_latitude
m = satellite_state.observe(moon).apparent()
s = satellite_state.observe(sun).apparent()
print(
    "What angle in the Sky is the crescent moon: "
    + str(position_angle_of(m.radec(), s.radec()))
)

[<img src="./img/1920px-Moon_phases_by_latitude.png" width="400"/>](./img/1920px-Moon_phases_by_latitude.png)

In [ ]:
sun = eph["sun"]
# Calculate the moon phase from the satellite perspective
fraction = moon_observation.fraction_illuminated(sun)
print(
    "What is the fraction of the moon illuminated by the sun: %f percent "
    % (100 * fraction)
)

# Orientation of Line-of-Sight relative to Satellite Velocity 

In [ ]:
from skyfield.constants import (
    AU_M,
    ANGVEL,
    DAY_S,
    DEG2RAD,
    ERAD,
    IERS_2010_INVERSE_EARTH_FLATTENING,
    RAD2DEG,
    T0,
    tau,
)
from skyfield.functions import dots

from numpy import (
    abs,
    arcsin,
    arccos,
    arctan2,
    array,
    clip,
    cos,
    minimum,
    pi,
    sin,
    sqrt,
    tan,
    where,
    zeros_like,
)

from skyfield.api import wgs84

# Convert ERAD to km
ERAD = ERAD * 10 ** -3
# Max earth radius at the equator
print("Max Earth Radius is", ERAD, "m")
print(
    "Distance of the satellite to earth center:",
    satellite_state.observe(earth).distance().km,
)
satellite_geocentric = wgs84.geographic_position_of(satellite.at(t0))
print(
    "The elevation of the Satellite above see level is",
    satellite_geocentric.elevation.km,
    "km",
)

# alpha is angle between measurement limb tangent point and observing satellite seen from the earth center
# beta is the angle between satellite limb_los and earth center
LOS_limb_pointing_elevation = 90

alpha = arccos(
    (ERAD + LOS_limb_pointing_elevation) / satellite_state.observe(earth).distance().km
)
print(
    "What is the pitch of satellite to point at an altitude of %f km : %f degree"
    % (LOS_limb_pointing_elevation, alpha * 180 / pi)
)

print(
    "Observation distance:",
    sqrt(
        (ERAD + LOS_limb_pointing_elevation)**2 + satellite_state.observe(earth).distance().km**2
    ),
    "km",
)

![](img/earth_limb_pitch.png)

In [14]:
#dd https://stackoverflow.com/questions/47319238/python-plot-3d-vectors

import plotly.graph_objs as go
def vector_plot(tvects,is_vect=True,orig=[0,0,0]):
    """Plot vectors using plotly"""

    if is_vect:
        if not hasattr(orig[0],"__iter__"):
            coords = [[orig,np.sum([orig,v],axis=0)] for v in tvects]
        else:
            coords = [[o,np.sum([o,v],axis=0)] for o,v in zip(orig,tvects)]
    else:
        coords = tvects

    data = []
    for i,c in enumerate(coords):
        X1, Y1, Z1 = zip(c[0])
        X2, Y2, Z2 = zip(c[1])
        vector = go.Scatter3d(x = [X1[0],X2[0]],
                              y = [Y1[0],Y2[0]],
                              z = [Z1[0],Z2[0]],
                              marker = dict(size = [0,5],
                                            color = ['blue'],
                                            line=dict(width=5,
                                                      color='DarkSlateGrey')),
                              name = 'Vector'+str(i+1))
        data.append(vector)

    layout = go.Layout(
             margin = dict(l = 4,
                           r = 4,
                           b = 4,
                           t = 4)
                  )
    fig = go.Figure(data=data,layout=layout)
    fig.show()

# Simulation of entire Missions

In [ ]:
import datetime

ts = load.timescale()
months = 12
time_range = range(0, 60 * 60 * 24 * 30 * months, 60)
t_delta = ts.utc(2025, 1, 1, 1, 1, time_range)

print("Start Time", t_delta[0].utc_strftime())
print("End Time", t_delta[-1].utc_strftime())
print("Delta t in seconds:", datetime.timedelta((t_delta[1] - t_delta[0])).seconds)
print("Total number of time steps:", len(t_delta))

## Transformation Tree between Measurement Data and Earth Coordiantes 
![](./img/tf_satellite.png)

## Create Observation Vectors
![](img/FOV_Moon_sun_venus.png)

In [17]:
# docs: Alternative https://rhodesmill.org/skyfield/api-position.html#skyfield.positionlib.ICRF.separation_from
from math import pi
from skyfield.api import load
from skyfield.functions import angle_between
from skyfield import framelib


eph = load("de421.bsp")
sun, venus, earth = eph["sun"], eph["venus"], eph["earth"]
sun_vector, sun_speed = (
    satellite_vector.at(t_delta)
    .observe(sun)
    .apparent()
    .frame_xyz_and_velocity(framelib.itrs)
)
moon_vector, moon_speed = (
    satellite_vector.at(t_delta)
    .observe(moon)
    .apparent()
    .frame_xyz_and_velocity(framelib.itrs)
)
earth_vector, earth_speed = (
    satellite_vector.at(t_delta)
    .observe(earth)
    .apparent()
    .frame_xyz_and_velocity(framelib.itrs)
)

venus_vector, venus_speed = (
    satellite_vector.at(t_delta)
    .observe(venus)
    .apparent()
    .frame_xyz_and_velocity(framelib.itrs)
)

## Define a LVLH Satellite fixed Reference Frame

LVLH (Local Vertical, Local Horizontal) - It is a rotating reference frame that is described by:

- the X axis is toward the velocity vector
- the Z axis is along the orbit normal pointing to the earth center
- the Y axis is the cross product between X and Z 
- Origin: Center of the Spacecraft

[<img src="./img/LVLH-Local-Vertical-Local-Horizontal-frame-definition.png" width="400"/>](./img/LVLH-Local-Vertical-Local-Horizontal-frame-definition.png)

In [18]:
from sklearn import preprocessing

#source: https://space.stackexchange.com/questions/48796/local-observer-coordinate-system-at-satellite-panel-lvlh-coordinate-system
#calculate LVLH reference frame for the reference sat
#Z = - R / ||R||
#Y = Z X V / ||Z X V||
#X = Y X Z
R = satellite.at(t_delta).position.km.T
V = satellite.at(t_delta).velocity.km_per_s.T
Z = -preprocessing.normalize(R, norm='l2')
Y = preprocessing.normalize(np.cross(Z, V), norm='l2')
X = np.cross(Y, Z)

In [19]:
# Rotate LOS
from pyquaternion import Quaternion

los_vector = []
rotated_vector=[]
angle = -pi+alpha
for x, y in zip(X, Y):
    rotated_vector.append(Quaternion(axis=y,angle=angle).rotate(x))
los_vector_x = [item[0] for item in rotated_vector]
los_vector_y = [item[1] for item in rotated_vector]
los_vector_z = [item[2] for item in rotated_vector]
los_vector_test = list(zip(los_vector_x, los_vector_y, los_vector_z))
los_vector_test = np.asarray(los_vector_test)

In [20]:
# Rotate Earth Vector



In [ ]:
vector_plot([X[0],Y[0],Z[0],los_vector_test[0]])
print("Angle between LOS and Speed Vector in degree", angle_between(los_vector_test[0],X[0])*(180/pi))
print("Angle between theoretical pitch vector", abs(angle*(180/pi)))
for count in range(len(los_vector_test)):
    if abs(angle_between(los_vector_test[count],X[count]) - abs(angle)) >= 16e-5:
        print(angle_between(los_vector_test[count],X[count]) - abs(angle))
        raise ValueError('Rotation failed')

[<img src="./img/LVLH-Local-Vertical-Local-Horizontal-frame-definition.png" width="400"/>](./img/LVLH-Local-Vertical-Local-Horizontal-frame-definition.png)

In [22]:
los_vector = list([los_vector_x, los_vector_y, los_vector_z])
los_vector = np.asarray(los_vector)

In [23]:
theta_moon_los = angle_between(moon_vector.km,los_vector) *(180/pi)
theta_moon_sun = angle_between(sun_vector.km, moon_vector.km) *(180/pi)
theta_sun_los = angle_between(sun_vector.km, los_vector) *(180/pi)
theta_venus_los = angle_between(venus_vector.km, los_vector) *(180/pi)

In [24]:
venus_blocked = satellite_vector.at(t_delta).observe(venus).apparent().is_behind_earth()
alt_venus, az_venus, distance_venus, alt_rate_venus, az_rate_venus, range_rate_venus = (
    satellite_vector.at(t_delta)
    .observe(venus)
    .apparent()
    .frame_latlon_and_rates(wgs84.geographic_position_of(satellite.at(t_delta)))
)

In [25]:
sunlit = satellite.at(t_delta).is_sunlit(eph)

In [26]:
moon_fraction_illuminated = (
    satellite_vector.at(t_delta).observe(moon).fraction_illuminated(sun)
)
moon_blocked = satellite_vector.at(t_delta).observe(moon).apparent().is_behind_earth()
ra, dec, distance = satellite_vector.at(t_delta).observe(moon).apparent().radec()
apparent_diameter_moon = Angle(radians=np.arcsin(radius_moon_km / distance.km) * 2.0)

In [27]:
alt_moon, az_moon, distance_moon, alt_rate_moon, az_rate_moon, range_rate_moon = (
    satellite_vector.at(t_delta)
    .observe(moon)
    .apparent()
    .frame_latlon_and_rates(wgs84.geographic_position_of(satellite.at(t_delta)))
)

In [28]:
from skyfield.api import wgs84
satellite_subpoint = wgs84.subpoint_of(satellite.at(t_delta))

## Analyse Simulated Data

In [29]:
import pandas as pd

df_moon = pd.DataFrame()
df_moon["timestamp"] = t_delta.utc_iso()
df_moon["theta_moon_los"] = theta_moon_los
df_moon["theta_moon_sun"] = theta_moon_sun
df_moon["theta_sun_los"]= theta_sun_los
df_moon["sunlit"] = sunlit
df_moon["delta_t"] = time_range
df_moon["moon_blocked"] = moon_blocked
df_moon["alt_rate.arcseconds.per_second"] = alt_rate_moon.arcseconds.per_second
df_moon["az_rate.arcseconds.per_second"] = az_rate_moon.arcseconds.per_second
df_moon["fraction_illuminated"] = moon_fraction_illuminated
df_moon["apparent_diameter_moon.degrees"] = apparent_diameter_moon.degrees
df_moon["satellite_subpoint"] = satellite_subpoint

In [ ]:
#df_moon[(df_moon["theta_moon_los"] < (FOV / 2.0)) & (df_moon["moon_blocked"] == False)]

In [ ]:
max_allowed_pitch_angle = FOV / 2
df_moon[(df_moon["theta_moon_los"] < max_allowed_pitch_angle) & (df_moon["sunlit"] == False) & (df_moon["moon_blocked"] == False)]

In [ ]:
max_allowed_pitch_angle = 100
df_moon[(df_moon["sunlit"] == False) & (df_moon["theta_moon_los"] < max_allowed_pitch_angle)]

In [ ]:
df_moon[(df_moon["moon_blocked"] == False)]

In [ ]:
df_moon[(df_moon["sunlit"] == False) & (df_moon["moon_blocked"] == False) & (df_moon["fraction_illuminated"]>0.75) & (df_moon["theta_moon_los"] < 20) ]

In [ ]:
import plotly.express as px

max_allowed_rotation_angle = 100
df_moon_ops = df_moon
fig1 = px.scatter(df_moon_ops, x="timestamp", y="theta_moon_los")

fig1.update_xaxes(title="Time")
fig1.update_yaxes(title="Angle between LOS and Moon in Degree")
fig1.update_layout(
    font_size=16,
    title=satellite_name+" showing all data"
)
#fig1.update_layout(yaxis_range=[0,1])

fig1.show()

In [ ]:
import plotly.express as px

max_allowed_rotation_angle = 360
min_faction_illuminated = 0.5
df_moon_ops = df_moon[(df_moon["moon_blocked"] == False) & (df_moon["fraction_illuminated"]>min_faction_illuminated) & (df_moon["theta_moon_los"] < max_allowed_rotation_angle) & (df_moon["theta_sun_los"] > 30) ]
fig1 = px.scatter(df_moon_ops, x="timestamp", y="theta_moon_los")

fig1.update_xaxes(title="Time")
fig1.update_yaxes(title="Angle between LOS and Moon in Degree")
fig1.update_layout(
    font_size=16,
    title=satellite_name+" - Unblocked and Illuminated: "+str(min_faction_illuminated)
)
fig1.show()

In [ ]:
import plotly.express as px

max_allowed_rotation_angle = 45
min_faction_illuminated = 0.20
max_faction_illuminated = 0.80
df_moon_ops = df_moon[(df_moon["moon_blocked"] == False) & (df_moon["fraction_illuminated"]>min_faction_illuminated) & (df_moon["fraction_illuminated"]< max_faction_illuminated) & (df_moon["theta_moon_los"] < max_allowed_rotation_angle) & (df_moon["theta_sun_los"] > 39/2) ]
fig1 = px.scatter(df_moon_ops, x="timestamp", y="theta_moon_los",template="simple_white")

fig1.update_xaxes(title="Time")
fig1.update_yaxes(title="Theta [deg]")
fig1.update_layout(
    font_size=16,
    title="Angle between nominal LOS and Moon Sight Vector for a Sun-Synchronous Orbit"
)
fig1.update_traces(marker_size = 4, marker_color="black")

fig1.show()

In [ ]:
import plotly.express as px

max_allowed_rotation_angle = 100
df_moon_ops = df_moon[(df_moon.index < 8000)  &  (df_moon["moon_blocked"] == False) & (df_moon["fraction_illuminated"]>0.25) & (df_moon["theta_moon_los"] < 360) & (df_moon["theta_sun_los"] > 30) ]
fig5 = px.scatter(df_moon_ops, x="timestamp", y="theta_moon_los")

fig5.update_xaxes(title="Time")
fig5.update_yaxes(title="Angle between LOS and Moon in Degree")
fig5.update_layout(
    font_size=16,
    title=satellite_name+" - Unblocked and Illuminated: "+str(min_faction_illuminated)
)
fig5.show()

In [39]:
import pandas as pd

df_venus = pd.DataFrame()
df_venus["theta_venus_los"] = theta_venus_los
df_venus["sunlit"] = sunlit
df_venus["delta_t"] = time_range
df_venus["venus_blocked"] = venus_blocked
df_venus["alt_rate.arcseconds.per_second"] = alt_rate_venus.arcseconds.per_second
df_venus["az_rate.arcseconds.per_second"] = az_rate_venus.arcseconds.per_second
df_venus["timestamp"] = t_delta.utc_iso()

In [ ]:
import plotly.express as px

fig = px.scatter(df_venus, x="timestamp", y="theta_venus_los")
fig.update_xaxes(title="timestamp")
fig.update_yaxes(title="Angle between LOS and Venus in Degree")
fig.update_layout(
    font_size=16,
    title=satellite_name+" showing all data"
)
fig.show()

In [ ]:
df_venus[(df_venus["sunlit"] == False) & (df_venus["venus_blocked"] == False)]

In [ ]:
max_allowed_pitch_angle = FOV / 2
df_venus[(df_venus["theta_venus_los"] < max_allowed_pitch_angle) & (df_venus["sunlit"] == False) & (df_venus["venus_blocked"] == False)]

## Stellar Rotating Rates relative to the Stellar Targets

In [ ]:
import plotly.express as px
fig = px.histogram(df_moon, x="az_rate.arcseconds.per_second")
fig.show()

In [ ]:
df_moon['az_rate.arcseconds.per_second'].abs().median()

In [ ]:
df_venus['alt_rate.arcseconds.per_second'].abs().median()

In [ ]:
df_venus['az_rate.arcseconds.per_second'].abs().median()

In [ ]:
import plotly.express as px

max_pitch_venus = 100
df_venus_ops = df_venus[
    (df_venus["venus_blocked"] == False)
    & (df_venus["theta_venus_los"] < max_pitch_venus)
    & (df_venus["sunlit"] == False)
]
fig2 = px.scatter(
    df_venus_ops, x="timestamp", y="theta_venus_los", color_discrete_sequence=["red"], 
)
fig.update_xaxes(title="timestamp")
fig.update_yaxes(title="Angle between LOS and Venus in Degree")
    
fig2.update_layout(font_size=16, title=satellite_name + "- Venus not blocked and no sunlit on the satellite")

fig2.show()
df_venus_ops

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure(data = fig1.data + fig2.data, )
fig.update_xaxes(title="Time")
fig.update_yaxes(title="Angle between LOS and Target in Degree")
fig.update_layout(
    font_size=16
)

fig.show()

In [49]:
from skyfield.api import wgs84
satellite_subpoint = wgs84.subpoint_of(satellite.at(t_delta[0]))

## Sun Exlustion Angle 

In [ ]:
sun_exclusion_angle = 39
df_sun_in_exlusion_angle = df_moon[(df_moon["sunlit"] == True) & (df_moon["theta_sun_los"] < sun_exclusion_angle/2)]
print("Number of events looking directly at the sun:",len(df_sun_in_exlusion_angle.index))

In [ ]:
fig3 = px.scatter(df_sun_in_exlusion_angle, x="timestamp", y="theta_sun_los",color_discrete_sequence=['red'])
fig3.update_xaxes(title="Time")
fig3.update_yaxes(title="Angle between LOS and Sun in Degree")
fig3.update_layout(
    font_size=16,
    title=satellite_name+ " showing all data"
)
fig3.show()


In [ ]:
sun_exclusion_angle = 39
df_sun_moon_in_exlusion_angle = df_moon[((df_moon["moon_blocked"]==False) & (df_moon["theta_moon_los"] < sun_exclusion_angle/2)) | ((df_moon["sunlit"] == True) & (df_moon["theta_sun_los"] < sun_exclusion_angle/2))]
print("Number of with violated sun exclusion angle by sun or moon:",len(df_sun_moon_in_exlusion_angle.index))

In [ ]:
fig3 = px.scatter(df_sun_in_exlusion_angle, x="timestamp", y="theta_sun_los",color_discrete_sequence=['red'])
fig3.update_xaxes(title="Time")
fig3.update_yaxes(title="Angle between LOS and Sun in Degree")
fig3.update_layout(
    font_size=16,
    title=satellite_name+ " showing all data"
)
fig3.show()

In [ ]:
print("The duty cycle at which sun exclusion angle is violated by the sun or moon in percent:", round(len(df_sun_moon_in_exlusion_angle.index)/len(t_delta)*100,5) )